# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:   <br>
**JOSÉ MARÍA LARIOS PÉREZ** <br>
**JORGE VARA RODRIGUEZ**<br>
Url: ...

Problema:

2. Organizar los horarios de partidos de La Liga<br>

Descripción del problema: Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar unalgoritmo que realice la asignación de los partidos a los horarios de forma que **maximice la audiencia.** 

(*) La respuesta es obligatoria                                  

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta

**Sin tener en cuenta las restricciones:** Al no tener restricciones estamos ante un problema de combinatoria de <u>combinación</u> (no hay restricciones, el orden NO importa) <u>con repetición</u> (no existe penalización por repetir franjas horarias). La fórmula según la cual se define este tipo de problemas es la siguiente: $CR_{n,r} = \binom{n+r-1}{r} = \frac{(n+r-1)!}{r! \cdot (n-1)!}$  

$$\displaystyle CR_{10,10} = \binom{10+10-1}{10} = \binom{19}{10} = \frac{19!}{10! \cdot 9!}$$

**Resultado:** 92,378 combinaciones posibles.

**Teniendo en cuenta las restricciones:** Al tener en cuenta las restricciones, el problema cambia a ser de tipo <u>variación</u> (en este caso, el orden sí que importa, ya que no es lo mismo que un equipo de tipo x juegue en una determinada franja horaria o en otra) y, aunque el problema establece una determinada penalización para el caso de disputar varios partidos en una franja horaria, el problema no impide dicha repetición, por lo que lo consideramos <u>con repetición.</u>  
**Fórmula:**
$$\displaystyle VR_{10,10} = 10^{10}$$

**Resultado:** $\displaystyle 10,000,000,000$ combinaciones posibles.


Como se puede observar, al importar el orden de los partidos, la cantidad de posibilidades crece exponencialmente respecto a las 92,378 del caso anterior.

Falta una restricción por añadir. El problema nos especifica que tanto el viernes como el lunes se tiene que jugar al menos un partido. Para hacer el cálculo final de opciones reciclamos la solución anterior y le aplicamos la diferencia de dejar viernes o lunes sin partido, añadiéndole las opciones de quedar ambos días sin partido para corregir la resta de casos ya quitados.  
- Dejar viernes o lunes son partido: $$\displaystyle 2 \cdot 9^{10} = 2 \cdot 3,486,784,401 = {6,973,568,802}$$
- Dejar ambos días sin partido: $$\displaystyle 8^{10} = 1,073,741,824$$
Por tanto quedaría de la siguiente manera.
**Resultado Final:** $$\displaystyle 10,000,000,000 - 6,973,568,802 + 1,073,741,824 = \mathbf{4,100,173,022}$$

Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

El uso de diccionarios es la mejor elección porque mejora la operabilidad y la claridad del código. Permite acceder a los datos por nombres descriptivos ('categoria', 'id') en lugar de por índices numéricos (posiciones), lo que hace que las funciones audiencia y resolver sean más fáciles de leer y mantener. Esta flexibilidad también significa que podemos añadir nuevos datos a un partido en el futuro sin romper la lógica actual del programa.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

Definición de la Función Objetivo ($A_T$)

El modelo busca **maximizar** la audiencia total del fin de semana. La audiencia de cada partido es dinámica: parte de un valor base según la importancia del encuentro, se ajusta según el horario elegido y se penaliza si coincide con otros partidos.

$$\max A_{T}=\sum _{i=1}^{10}\left(\text{Penalización}(k_{i})\cdot \sum _{j=1}^{k_{i}}(\text{Base}_{j}\cdot \text{Coef}_{i})\right)$$

| Símbolo | Concepto | Definición |
| :--- | :--- | :--- |
| **$A_T$** | **Audiencia Total** | El valor final acumulado que el modelo intenta maximizar. |
| **$i$** | **Índice de Franja** | Representa cada una de las 10 franjas horarias (Viernes, Sábados, Domingos, Lunes). |
| **$j$** | **Índice de Partido** | Representa cada uno de los 10 partidos individuales de la jornada. |
| **$k_i$** | **Densidad** | Número de partidos totales asignados a la franja $i$. |
| **$\text{Base}_j$** | **Audiencia Intrínseca** | Audiencia potencial del partido $j$ según las categorías de los equipos (A, B o C). Es el valor "puro" del partido. |
| **$\text{Coef}_i$** | **Coeficiente de Franja** | Factor de corrección (0 a 1) que ajusta la audiencia según el atractivo del horario. |
| **$\text{Penalización}(k_i)$** | **Factor de Solape** | Coeficiente que reduce la audiencia si hay concurrencia ($k_i > 1$) para simular la competencia entre canales. |

El objetivo es **maximizar** la suma de las audiencias de todos los partidos, donde la audiencia de cada partido depende de la categoría de los equipos que se enfrentan, el horario elegido y si compite con otros partidos simultáneamente.

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [7]:
import itertools

# Liga de 14 equipos para hacer pruebas
PARTIDOS_FB = [
    {"id": "Elche-Osasuna", "categoria": ("C", "B")},
    {"id": "Espanyol-Celta", "categoria": ("B", "B")},
    {"id": "Getafe-Villarreal", "categoria": ("B", "B")},
    {"id": "Sevilla-Alavés", "categoria": ("B", "C")},
    {"id": "Real Madrid-R. Sociedad", "categoria": ("A", "B")},
    {"id": "Oviedo-Athletic", "categoria": ("C", "B")},
    {"id": "Rayo-Atlético", "categoria": ("C", "A")},
]

COEF_HORARIO = {
    "V20": 0.4, "S12": 0.55, "S16": 0.7, "S18": 0.8, "S20": 1.0,
    "D12": 0.45, "D16": 0.75, "D18": 0.85, "D20": 1.0, "L20": 0.4
}

PENALIZACION_SOLAPE = {
    1: 1.0, 2: 0.75, 3: 0.55, 4: 0.4, 5: 0.3, 6: 0.25, 7: 0.22, 8: 0.2
}

AUDIENCIA_BASE = {
    ("A", "A"): 2.00, ("A", "B"): 1.30, ("B", "B"): 0.90,
    ("A", "C"): 1.00, ("B", "C"): 0.75, ("C", "C"): 0.50
}

NUM_PARTIDOS = len(PARTIDOS_FB)

In [8]:
def audiencia(calendario):
    audiencia_total = 0
    for horario, partidos in calendario.items():

        if not partidos:
            continue

        coef_horario = COEF_HORARIO[horario]
        num_solapes = len(partidos)
        factor_penalizacion = PENALIZACION_SOLAPE.get(num_solapes, 0.2)

        audiencia_bruta = 0
        for partido in partidos:

            # Esta linea evita que pueda haber algo como D-A y A-D
            categoria = tuple(sorted(partido['categoria']))
            valor_base = AUDIENCIA_BASE[categoria]
            audiencia_bruta += valor_base * coef_horario

        audiencia_total += audiencia_bruta * factor_penalizacion
    return audiencia_total

# Esta función generá un diccionario donde se puede ver los partidos /
# que están asociados a cada horario
def convertir_a_diccionario(asignacion, partidos):
    calendario = {horario: [] for horario in list(COEF_HORARIO.keys())}
    for i, horario in enumerate(asignacion):
        calendario[horario].append(partidos[i])
    return calendario

# Esta función genera todas las posibles conbinaciones de horarios
def generar_asignaciones(num_partidos, horario_ids):
    opciones = itertools.product(horario_ids, repeat=num_partidos)
    # Todas las asignaciones que al menos incluyen una vez V20 y L20
    for asignacion in opciones:
        if "V20" in asignacion and "L20" in asignacion:
            yield asignacion # No es una lista para no saturar la memoria


# Esta funcion simplemente es para mostrar los resultados de forma más legible
def mostrar_mejor_calendario(calendario, audiencia_total):
    print(f"\n{'HORARIO':<10} | {'PARTIDOS'}")
    print("=" * 50)
    for horario in list(COEF_HORARIO.keys()):
        partidos_lista = calendario.get(horario, [])
        if partidos_lista:
            nombres = ", ".join(p['id'] for p in partidos_lista)
        else:
            nombres = "—"
        print(f"  {horario:<8} | {nombres}")
    print("=" * 50)
    print(f"  AUDIENCIA TOTAL: {audiencia_total:.2f} millones\n")

In [9]:
# Por defecto, hay puestos 14 equipos (20 equipos tardaba demasiado)

mejor_audiencia = 0
mejor_calendario = {}
horarios_id = list(COEF_HORARIO.keys())

for asignacion in generar_asignaciones(NUM_PARTIDOS, horarios_id):
    calendario_actual = convertir_a_diccionario(asignacion, PARTIDOS_FB)

    # Por cada calendario creado, calculamos su audiencia
    audiencia_actual = audiencia(calendario_actual)
    if audiencia_actual > mejor_audiencia:
        mejor_audiencia = audiencia_actual
        mejor_calendario = calendario_actual

mostrar_mejor_calendario(mejor_calendario, mejor_audiencia)


HORARIO    | PARTIDOS
  V20      | Elche-Osasuna
  S12      | —
  S16      | —
  S18      | Espanyol-Celta
  S20      | Real Madrid-R. Sociedad
  D12      | —
  D16      | Sevilla-Alavés
  D18      | Getafe-Villarreal
  D20      | Rayo-Atlético
  L20      | Oviedo-Athletic
  AUDIENCIA TOTAL: 4.95 millones



Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

Para determinar la eficiencia del algoritmo, se analiza el coste computacional de cada función de manera individual y su impacto en el flujo principal de ejecución.

- Función **convertir_a_diccionario**:  
  **Complejidad:** **$O(N + M)$**. Es una función de coste lineal.

- Función **audiencia**:  
  **Complejidad:** **$O(N)$**. Su ejecución escala de forma lineal respecto al número de partidos.

- Función **generar_asignaciones**:  
  **Complejidad:** **$O(N \cdot K^N)$**. Es una complejidad de tipo **exponencial**.

La complejidad total del algoritmo es el producto del número de combinaciones generadas por el coste de procesar cada una de ellas dentro del bucle principal, lo que nos queda:

**Complejidad Temporal Total: $O(K^N)$**

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

In [10]:
import random
import math

def calendario(
    partidos,
    horario_ids,
    temp = 100000000,
    temp_min = 0.001,
    enfriamiento = 0.999
):

    actual_calendario= [random.choice(horario_ids) for _ in range(len(partidos))]

    # Dado que tenemos la restricción de que V20 y L20 tienen que esta al menos una vez /
    # forzamos su inclusión
    if "V20" not in actual_calendario: actual_calendario[0] = "V20"
    if "L20" not in actual_calendario: actual_calendario[1] = "L20"

    mejor_calendario = actual_calendario
    mejor_audiencia = audiencia(convertir_a_diccionario(actual_calendario, partidos))

    while temp > temp_min:
        idx_partido = random.randrange(len(partidos))
        antiguo_horario = actual_calendario[idx_partido]
        nuevo_horario = random.choice(horario_ids)

        if antiguo_horario == nuevo_horario:
            temp *= enfriamiento
            continue

        actual_calendario[idx_partido] = nuevo_horario

        # tenemos en cuenta la restricción de que V20 y L20 tienen que estar al menos una vez
        if "V20" not in actual_calendario or "L20" not in actual_calendario:
            actual_calendario[idx_partido] = antiguo_horario
            continue

        audiencia_actual = audiencia(convertir_a_diccionario(actual_calendario, partidos))
        diferencia = audiencia_actual - mejor_audiencia

        if diferencia > 0 or random.random() < math.exp(diferencia / temp):
            if audiencia_actual > mejor_audiencia:
                mejor_audiencia = audiencia_actual
                mejor_calendario = list(actual_calendario)
        else:
            actual_calendario[idx_partido] = antiguo_horario

        temp *= enfriamiento

    return convertir_a_diccionario(mejor_calendario, partidos), mejor_audiencia

Estos son algunos puntos donde se puede ver la ventaja del recocido frente a la fuierza bruta:
### Eficiencia Temporal:
**- Fuerza Bruta:** Tiene una complejidad exponencial $O(K^N)$. Para 10 partidos y 10 franjas, existen $10^{10}$ (diez mil millones) de combinaciones. 

**- Recocido Simulado:** Tiene una complejidad lineal respecto a las iteraciones $O(I \cdot N)$. Se resuelve en milisegundos realizando apenas unas 9,000 evaluaciones.

### Evasión de Máximos Locales:
El recocido simulado permite aceptar soluciones peores al principio (con alta temperatura) para saltar mínimos locales y no quedar atrapado en un pico de audiencia que este lejos de un resultado más cerca del óptimo.

### Escalabilidad:
Si hubiese 20 partidos, la fuerza bruta pasaría a $10^{20}$ combinaciones (imposible de computar). El recocido simulado simplemente duplicaría su tiempo de ejecución lineal, manteniendo la viabilidad del problema.

(*)Calcula la complejidad del algoritmo

Respuesta

A diferencia del modelo exponencial anterior, este algoritmo basado en Recocido Simulado (Simulated Annealing) es de complejidad controlada, lo que permite obtener soluciones óptimas en milisegundos.  

**- Inicialización:** La generación del estado inicial y la verificación de las restricciones de obligatoriedad (V20 y L20) tiene un coste de $O(N)$.  
**- Bucle Principal (Enfriamiento):** El número de iteraciones ($I$) es finito y viene determinado por el descenso de la temperatura. Con $\alpha = 0.999$, se realizan aproximadamente 9,210 iteraciones.  
**- Generación de Vecinos:** En cada paso, seleccionar un partido al azar y cambiar su franja tiene un coste de $O(1)$. La validación de la restricción (verificar que V20 y L20 no queden vacíos) tiene un coste de $O(K)$.  
**- Cálculo de Audiencia:** Dentro de cada iteración, se invoca la función audiencia para evaluar el nuevo estado, la cual recorre los partidos y franjas con un coste de $O(N + K)$.  
**- Complejidad Total:** $O(I \cdot (N + K))$  
**Nota:** Dado que el número de iteraciones ($I$) y el número de franjas ($K$) son constantes fijadas por la configuración del problema, la complejidad efectiva del algoritmo es lineal $O(N)$ respecto al número de partidos. Esto evita la explosión combinatoria del enfoque de fuerza bruta original.

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

In [11]:
# Definición de equipos por categorías jornada 24 de la temporada 25/26:
# A (Top): Real Madrid, FC Barcelona, Atlético de Madrid
# B (Medio): Sevilla, Athletic, Valencia, Real Sociedad, Villarreal, Betis, Celta, Osasuna, Girona, Espanyol, Getafe
# C (Bajo): Elche, Alavés, Oviedo, Rayo, Levante, Mallorca

PARTIDOS_RS = [
    {"id": "Elche-Osasuna", "categoria": ("C", "B")},
    {"id": "Espanyol-Celta", "categoria": ("B", "B")},
    {"id": "Getafe-Villarreal", "categoria": ("B", "B")},
    {"id": "Sevilla-Alavés", "categoria": ("B", "C")},
    {"id": "Real Madrid-R. Sociedad", "categoria": ("A", "B")},
    {"id": "Oviedo-Athletic", "categoria": ("C", "B")},
    {"id": "Rayo-Atlético", "categoria": ("C", "A")},
    {"id": "Levante-Valencia", "categoria": ("C", "B")},
    {"id": "Mallorca-Betis", "categoria": ("C", "B")},
    {"id": "Girona-Barcelona", "categoria": ("B", "A")}
]

Aplica el algoritmo al juego de datos generado

Respuesta

In [12]:
horario_ids = list(COEF_HORARIO.keys())

# Ejecución del recocido simulado
mejor_cal, mejor_aud = calendario(PARTIDOS_RS, horario_ids)

mostrar_mejor_calendario(mejor_cal, mejor_aud)


HORARIO    | PARTIDOS
  V20      | Mallorca-Betis
  S12      | Rayo-Atlético
  S16      | Sevilla-Alavés
  S18      | Espanyol-Celta
  S20      | Girona-Barcelona
  D12      | Levante-Valencia
  D16      | Getafe-Villarreal
  D18      | Elche-Osasuna
  D20      | Real Madrid-R. Sociedad
  L20      | Oviedo-Athletic
  AUDIENCIA TOTAL: 6.65 millones



Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

Una linea interesante sería aplicarlo a competiciones con más partidos en cada jornada como puede ser la Champions League o, si se quiere extrapolar a otro terreno, se podría aplicar a organizar una sesión diaria de una disciplina de los juegos olimpicos en el que ya no se trabajaría entre dias sino entre horas de un mismo dia y con muchos más participantes que solo 10. El espacio de soluciones es mucho mayor por lo que algoritmicamente es un reto pero tambien puede ser más dificil tener el dato de audiencia o algo parecido. A nivel de algoritmos se puede probar con algunos que ofrezcan una mejor exploración.

Otra linea interesante sería añadir más restricciones y ya no solo restricciones fijas para todas las jornadas sino restricciones para una jornada especifica y ver como se adaptan los distintos algoritmos
